# （発展・任意）ステップ6：機械が決めた基準と、自分が決めた基準を比べる

これは探究講座の **任意** のステップです。時間が余った班・興味のある人向け。

ステップ3で、あなたは「日照率がこれ以下なら永久影だろう」というしきい値を **自分で** 決めました。
同じことを、**機械学習に学習させる** とどうなるでしょう。

- **やること**：「日照率」と「緯度」から、その場所が永久影かどうかを当てるモデルをつくる
- **モデルは5種類**：ロジスティック回帰／決定木／k近傍／ニューラルネット／ランダムフォレスト
- **仕組みは覚えなくてよい**。`method=` を取り替えて、境界線の形・当たりやすさ・読みやすさを比べる

## 進め方
1. まずワークシートに **予想** を書く（どれが一番当たる？　どれが一番「読める」？）
2. セルを実行して、5つのモデルの結果を見る
3. 5つを順位づけして、理由を書く

In [ ]:
from moonkit import *
from moonkit_ml import train, METHODS

# 南極のデータ。ラベル（正解）は「永久影かどうか」（permanent_shadow_fraction が 0.5 以上）
極 = south_pole(load('極域日照'))
ラベル = (極['permanent_shadow_fraction'] >= 0.5)
print('南極の地点数：', len(極), '／ そのうち永久影：', int(ラベル.sum()),
      f'（{ラベル.mean()*100:.1f}%）')

my_threshold = 5     # ★ここを変える：ステップ3であなたが決めた「日照率のしきい値」[%]

---
## 5つのモデルを全部ためす

下のセルを実行すると、5つのモデルそれぞれについて：
- **色つきの地図**：そのモデルが「ここは永久影」と判定する範囲（オレンジ）と、そうでない範囲（青）
- **点**：テストデータの正解（青＝永久影でない、オレンジ＝永久影）
- **赤い点線**：あなたがステップ3で決めたしきい値
- **正解率**：テストデータで何％当たったか
- **ルールを言葉で説明できる？**
が出ます。

In [ ]:
結果 = {}
for m in METHODS:
    r = train(極, features=['average_illumination_percent', 'lat'], label=ラベル,
              method=m, my_threshold=my_threshold)
    結果[m] = r['テスト正解率']
    print()

print('=== まとめ ===')
for m, acc in sorted(結果.items(), key=lambda kv: -kv[1]):
    print(f'  {m:12s} テスト正解率 {acc:.3f}')

**ワークシートに記録**：5つのモデルの正解率。境界線の形は5つでどう違った？
（まっすぐ／階段状／ぐにゃぐにゃ／なめらか／ブロック）

---
## 「複雑さ」を上げると何が起きる？（過学習の実験）

`複雑さ='高い'` にすると、k近傍は「となりの1点だけ」を見るようになり、
ニューラルネットは大きくなります。境界線がどうなるか見てみましょう。

In [ ]:
train(極, ['average_illumination_percent', 'lat'], ラベル,
      method='k近傍', 複雑さ='高い', my_threshold=my_threshold)

In [ ]:
train(極, ['average_illumination_percent', 'lat'], ラベル,
      method='ニューラルネット', 複雑さ='高い', my_threshold=my_threshold)

**気づいたこと**：境界線が「飛び地」だらけになっていませんか？
練習データの正解率は上がるのに、テストデータの正解率は上がらない（むしろ下がる）ことがあります。
これを **過学習（かがくしゅう）** といいます。「練習問題を丸暗記したけれど、本番の問題は解けない」状態です。

---
## 考える（ワークシートに書く）

1. **正解率で順位をつける**と？　1位と最下位の差は何ポイント？
2. **「ルールを言葉で説明できるか」で順位をつける**と？（1と順位は同じ？　ちがう？）
3. あなたがステップ3で決めたしきい値（`my_threshold`）の正解率は、機械のモデルと比べてどうだった？
   ゆるいしきい値（15% など）にしていた人は、「何も考えず多い方に賭ける」よりも悪くなっていない？
4. 決定木が見つけたルールを読んでみよう。あなたのしきい値と近い？　遠い？
5. **宇宙飛行士が住む場所を決めるとき**、正解率が少し高いけれど「なぜそう判定したか説明できない」モデルと、
   正解率が少し低いけれど「if〜then のルールが読める」モデル、どちらを使う？　なぜ？